# Exp 20: Coarse dropout on non-VIP images

Card: [exp20_coarse_dropout_non_vip.md](../../experiments/exp20_coarse_dropout_non_vip.md)

In [ ]:
import os

import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve

from core.config import CLASSES, ROOT
from core.cv import VIP, CVResult, Config, CrossValidator
from core.gradcam import GradCamFigures
from core.tracking import Experiment

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-coarse-dropout-non-vip-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_SAMPLE = 2  # Cat, Dog images per comparison figure
    N_POSITIVES = 2  # images per VIP Grad-CAM figure
else:
    OVERRIDES = {}
    N_SAMPLE = 16  # exp19 sample
    N_POSITIVES = None  # all labeled positives
SEED = 0  # exp19 sample
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp20_coarse_dropout_non_vip" / "resnet18.toml", **OVERRIDES
)
EXP16 = Config.from_toml(CONFIG_DIR / "exp16_downsampling" / "resnet18.toml")
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp20_coarse_dropout_non_vip"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SERIES = {"exp16": "#eb6834", "coarse dropout": "#2a78d6"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## Training

In [ ]:
results = {"exp16": CVResult.load(EXP16)}
results["coarse dropout"] = CrossValidator(CONFIG).run()
figures = {}

## Grad-CAM Cat, Dog: exp16 vs coarse dropout

In [ ]:
cams = {"exp16": GradCamFigures(EXP16), "coarse dropout": GradCamFigures(CONFIG)}
source = "coarse dropout" if SMOKE else "exp16"  # exp19 rows drawn from exp16 OOF; smoke OOF holds few rows
for cls in ["Cat", "Dog"]:
    rows = cams[source].positive_rows(cls, N_SAMPLE, SEED)
    figures[f"{PREFIX}_grad_cam_{cls.lower()}_compare.png"] = GradCamFigures.compare(cams, cls, rows)
plt.show()

## Grad-CAM VIP: unseen vs trained images

In [ ]:
for cls in VIP:
    for train in (False, True):
        fig = cams["coarse dropout"].positives(cls, train=train, n=N_POSITIVES)
        split = "train" if train else "validation"
        figures[f"{PREFIX}_grad_cam_{cls.lower()}_{split}.png"] = fig
plt.show()

## Precision-recall curves

In [ ]:
fig, axes = plt.subplots(
    1, len(CLASSES), figsize=(3.4 * len(CLASSES), 3.6), sharey=True
)
for cls, ax in zip(CLASSES, axes):
    for name, r in results.items():
        y, p = r.oof[cls], r.oof[f"p_{cls}"]
        precision, recall, _ = precision_recall_curve(y, p)
        ax.plot(
            recall,
            precision,
            color=SERIES[name],
            lw=2,
            drawstyle="steps-post",
            label=f"{name}, AP {average_precision_score(y, p):.3f}",
        )
    ax.set(
        title=f"{cls} ({int(y.sum())} positives)",
        xlabel="recall",
        xlim=(0, 1.02),
        ylim=(0, 1.05),
    )
    ax.legend(loc="lower left", alignment="left")
axes[0].set_ylabel("precision")
fig.tight_layout()
figures[f"{PREFIX}_pr_curves.png"] = fig
plt.show()

In [ ]:
with Experiment(CONFIG.experiment).run("summary", {"val_mode": "fast"}) as run:
    if SAVE_FIGURES:
        run.log_figures(figures, FIGURES)